# Track 4: Aligned Response Refinement with Direct Preference Optimization (DPO)

This notebook demonstrates how to align a model with preferred human feedback using Direct Preference Optimization (DPO) and `DPOTrainer` from Hugging Face `trl`. We use **Unsloth** for memory-efficient and fast LoRA training of the **Qwen2.5-3B-Instruct** model.

## 1. Setup Environment and Imports
We load our dependencies and identify the compute device capability to determine if bfloat16 is supported.

In [1]:
import os
import torch
import warnings
from datasets import load_dataset
from transformers import AutoTokenizer
from unsloth import FastLanguageModel, PatchDPOTrainer
from trl import DPOTrainer, DPOConfig

warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

compute_dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8 else torch.float16
print(f'Using device: cuda | Dtype: {compute_dtype}')

[unsloth.import_fixes|WARNING]Unsloth: Detected broken vLLM binary extension; disabling vLLM imports and continuing import.
Please reinstall via `uv pip install unsloth vllm torchvision torchaudio --torch-backend=auto`.


/home/lmassaron/code/sft-examples/.venv/lib/python3.12/site-packages/unsloth/__init__.py:1427: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W0721 12:17:03.369000 72976 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


W0721 12:17:03.384000 72976 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


🦥 Unsloth Zoo will now patch everything to make training faster!


/home/lmassaron/code/sft-examples/.venv/lib/python3.12/site-packages/unsloth/import_fixes.py:1200: FutureWarning: torch._dynamo.config.inline_inbuilt_nn_modules is deprecated and does not do anything, inline_inbuilt_nn_modules is always True. It will be removed in a future version of PyTorch.
  original_setattr(self, name, value)


Using device: cuda | Dtype: torch.bfloat16


## 2. Load Model & Enable PEFT
We load `unsloth/Qwen2.5-3B-Instruct-bnb-4bit` using Unsloth's optimized 4-bit precision to fit within small VRAM budgets, and attach LoRA adapters to all projection layers.

In [2]:
MODEL_ID = 'unsloth/Qwen2.5-3B-Instruct-bnb-4bit'

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=1024,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=8,
    lora_alpha=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=3407,
)
print('Model and PEFT adapters loaded successfully.')

==((====))==  Unsloth 2026.7.3: Fast Qwen2 patching. Transformers: 5.14.1. vLLM: 0.19.1.
   \\   /|    NVIDIA GB10. Num GPUs = 1. Max memory: 121.689 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.13.0+cu130. CUDA: 12.1. CUDA Toolkit: 13.0. Triton: 3.7.1
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Unsloth 2026.7.3 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


Model and PEFT adapters loaded successfully.


## 3. Load and Format Dataset (Orca DPO Pairs)
We load the `Intel/orca_dpo_pairs` dataset from Hugging Face, shuffle it, and select a small subset of 250 training examples and 50 validation examples. We map the inputs to a standard DPO format where the user prompt is compiled using the model's native chat template.

In [3]:
dataset = load_dataset('Intel/orca_dpo_pairs', split='train')
shuffled = dataset.shuffle(seed=42)
train_ds = shuffled.select(range(250))
eval_ds = shuffled.select(range(250, 300))

def format_dpo_example(example):
    messages = [{'role': 'user', 'content': example['question']}]
    prompt_str = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return {
        'prompt': prompt_str,
        'chosen': example['chosen'],
        'rejected': example['rejected']
    }

train_mapped = train_ds.map(format_dpo_example, remove_columns=train_ds.column_names)
eval_mapped = eval_ds.map(format_dpo_example, remove_columns=eval_ds.column_names)
print('Prompt Preview:\n', train_mapped[0]['prompt'])
print('Chosen Preview:\n', train_mapped[0]['chosen'])
print('Rejected Preview:\n', train_mapped[0]['rejected'])

Prompt Preview:
 <|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
This is some data: CBS PLAY-BY-PLAY Chris Schenkel (first half) and Ray Scott (second half); 1962 NETWORK CBS.

Generate a detailed description of this data<|im_end|>
<|im_start|>assistant

Chosen Preview:
 Okay, imagine you are watching a fun game on TV with your family. In this case, the game happened in 1962. Now, on TV, there are people who talk to us and tell us what is happening in the game. They help us understand the game better, just like how I'm helping you understand things right now.

In this data, there are two people who talked about the game in 1962. The first person, Chris Schenkel, talked about the game in the first half. The second person, Ray Scott, talked about the game in the second half. Both of them worked for a big TV company called CBS. So, this sentence is just telling us who talked about the game on TV and when they did it.
Rejec

## 4. Run Aligned DPO Fine-Tuning
We run Unsloth's patched version of the TRL `DPOTrainer` (`PatchDPOTrainer()`). We run training for 60 steps with a learning rate of 5e-6, using cosine annealing decay.

In [4]:
PatchDPOTrainer()

training_args = DPOConfig(
    output_dir='qwen2.5-3b-dpo-output',
    beta=0.1,
    max_length=1024,
    max_prompt_length=512,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-6,
    max_steps=60,
    lr_scheduler_type='cosine',
    warmup_ratio=0.1,
    bf16=(compute_dtype == torch.bfloat16),
    fp16=(compute_dtype == torch.float16),
    logging_steps=10,
    eval_strategy='steps',
    eval_steps=20,
    save_steps=20,
    report_to='none',
)

trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=training_args,
    train_dataset=train_mapped,
    eval_dataset=eval_mapped,
    processing_class=tokenizer,
)

model.config.use_cache = False
trainer.train()

model.save_pretrained('qwen2.5-3b-dpo-adapter')
tokenizer.save_pretrained('qwen2.5-3b-dpo-adapter')
print('DPO adapter successfully saved!')

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 250 | Num Epochs = 2 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 14,966,784 of 3,100,905,472 (0.48% trained)


`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!


...

Unsloth: Restored added_tokens_decoder metadata in qwen2.5-3b-dpo-output/checkpoint-20/tokenizer_config.json.


Unsloth: Restored added_tokens_decoder metadata in qwen2.5-3b-dpo-output/checkpoint-40/tokenizer_config.json.


Unsloth: Restored added_tokens_decoder metadata in qwen2.5-3b-dpo-output/checkpoint-60/tokenizer_config.json.


Unsloth: Restored added_tokens_decoder metadata in qwen2.5-3b-dpo-adapter/tokenizer_config.json.


DPO adapter successfully saved!


## 5. Evaluation and Inference comparison
We put the model in inference mode and query it with a test prompt to check if it adheres to Aligned Response preferences.

In [5]:
FastLanguageModel.for_inference(model)

test_question = "Explain why the sky is blue in one concise sentence."
messages = [{'role': 'user', 'content': test_question}]
inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors='pt').to('cuda')

with torch.no_grad():
    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=100,
        use_cache=True,
        pad_token_id=tokenizer.eos_token_id
    )

response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True).strip()
print('Question:', test_question)
print('\nAligned Response:', response)

Both `max_new_tokens` (=100) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: Explain why the sky is blue in one concise sentence.

Aligned Response: The sky appears blue because the Earth's atmosphere scatters sunlight more efficiently for the shorter blue wavelengths, making the blue light appear more prominent when we look at the sky.
